## Meet OpsPilot (the project you will grow)

**Meridian Supply Co.** is a fictional company that sells networking hardware to businesses.
Its support and finance staff spend their day answering questions such as *"Which plan is
customer C001 on?"*, *"Was order O1002 charged twice?"*, *"What does our refund policy say for
an Enterprise customer?"* and, sometimes, *"Refund this customer."*

**OpsPilot** is the assistant we build for them. It is not a product you can download; it is
the running example of this notebook. Every section adds one capability and every capability
is motivated by something the previous version could not do. The company data is deliberately
tiny and lives inside the notebook:

```text
CUSTOMERS   three customer records   (id, name, plan, email)          -> get_customer tool  (L3)
ORDERS      three orders             (customer, item, amount, status) -> get_order tool     (L3)
WEATHER     three cities                                             -> get_weather tool   (L3)
POLICY_DOCS refund, shipping and escalation policies                 -> search_policies    (L8)
REFUND_LEDGER  every refund the agent ever issues                    -> refund_customer    (L4)
```

Two kinds of people talk to OpsPilot: **support** staff, who may only read, and **finance**
staff, who may also move money. That difference drives the permission and approval sections.

## How to read the code cells

Three libraries appear next to our own code, and it is easy to lose track of which is which.
Comments in every code cell say where a thing comes from:

```python
model.invoke(messages)        # LangChain: invoke() = one request, one AIMessage
agent.get_state(config)       # LangGraph: read the saved checkpoint
class Ticket(BaseModel): ...  # Pydantic: data validation library used by LangChain
show_messages(result)         # ours: defined in this notebook
```

- **LangChain** gives you models, messages, tools, `create_agent()`, middleware and retrieval.
- **LangGraph** is the engine underneath: graphs, state, checkpoints, interrupts, streaming.
  `create_agent()` returns a LangGraph graph, which is why `invoke`, `stream`, `get_state`
  and `__interrupt__` on an *agent* are LangGraph features.
- **ours** means a function, class or data structure defined in this notebook, including the
  mock model used when you have no API key.

## Your API key (30 seconds)

The course model runs on OpenRouter. Give this notebook your issued key in one of two ways:

- **Recommended:** click the key icon in Colab's left sidebar, add a secret named
  `OPENROUTER_API_KEY`, and switch on *Notebook access*. Every course notebook then finds it automatically.
- **Or:** run the cell below and paste the key when asked (it is kept only in this session).

No key? Press Enter when asked. The notebook switches to the mock model and everything still runs.
Never paste a key into a code cell: notebooks get shared.

In [ ]:
# === Setup: run this cell first ===============================================
# Installs LangChain 1.x, reads your API key, and defines make_model(): the ONE
# function every section uses to obtain a chat model.
%pip install -q -U "langchain>=1.2" "langchain-openai>=1.1" "langgraph>=1.0" "langchain-text-splitters>=1.0"

import json, os, re, time                          # Python standard library
from getpass import getpass

MODEL_NAME = "openai/gpt-oss-120b"                 # the course model on OpenRouter
OPENROUTER_URL = "https://openrouter.ai/api/v1"

def load_api_key():                                 # ours
    """Look for the key in Colab Secrets, then the environment, then ask once."""
    try:
        from google.colab import userdata           # only exists on Colab
        key = userdata.get("OPENROUTER_API_KEY")
        if key:
            return key, "Colab secret"
    except Exception:
        pass                                        # not on Colab, or no secret yet
    if os.getenv("OPENROUTER_API_KEY"):
        return os.environ["OPENROUTER_API_KEY"], "environment variable"
    try:
        key = getpass("OpenRouter API key (press Enter to use the mock model): ").strip()
    except Exception:                               # no keyboard available (automated run)
        key = ""
    return (key, "typed in") if key else ("", "none")

API_KEY, KEY_SOURCE = load_api_key()
LIVE = bool(API_KEY)                                # True = real model, False = mock model

def make_model(temperature=0.0, model_name=MODEL_NAME, broken=False):   # ours
    """Return a LangChain chat model.

    LIVE  -> ChatOpenAI pointed at OpenRouter. LangChain's OpenAI integration speaks the
             OpenAI-compatible API, so only base_url and the model name change.
    MOCK  -> MockChatModel, a rule-based stand-in defined in the next cell.
    broken=True returns a model that always fails (used to demonstrate fallbacks).
    """
    if not LIVE:
        return MockChatModel(fail=broken)
    from langchain_openai import ChatOpenAI          # LangChain: chat model class for OpenAI-compatible APIs
    return ChatOpenAI(                                # LangChain: one object = one configured model
        model="openai/this-model-does-not-exist" if broken else model_name,
        api_key=API_KEY,
        base_url=OPENROUTER_URL,
        temperature=temperature,
        max_tokens=900,
        extra_body={"reasoning": {"effort": "low"}},   # keep hidden reasoning short and cheap
    )

print("Model  :", MODEL_NAME)
print("Key    :", KEY_SOURCE)
print("Mode   :", "LIVE - real model replies" if LIVE else "MOCK - canned replies, real shapes, zero cost")

### The mock model (run it; read it later, or never)

When there is no key, `make_model()` returns the class below. It is a real LangChain chat model
(`BaseChatModel` subclass) that answers this notebook's questions with fixed rules: it requests
tool calls when a question mentions a customer, an order, a city, a sum, a policy, and so on, and
otherwise replies with short canned text. Because it produces genuine `AIMessage` objects with
`tool_calls`, every LangChain mechanism in this notebook (agents, middleware, interrupts,
structured output, graphs) runs unchanged on top of it. You do not need to understand it now.

In [ ]:
from typing import Any, Optional
from langchain_core.language_models import BaseChatModel          # LangChain: base class of every chat model
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage   # LangChain: message classes
from langchain_core.outputs import ChatGeneration, ChatResult      # LangChain: what _generate must return
from langchain_core.utils.function_calling import convert_to_openai_tool   # LangChain: tool -> JSON schema

def text_of(message) -> str:                                       # ours
    """Message content as plain text (real models may return a list of content blocks)."""
    content = message.content
    if isinstance(content, str):
        return content
    return " ".join(block.get("text", "") for block in content if isinstance(block, dict))

def _phrase(name, content):                                        # ours (mock helper)
    """Turn one tool result into a readable sentence for the mock's final answer."""
    try:
        data = json.loads(content)
    except Exception:
        data = None
    if name == "calculate":
        return f"The result is {content}."
    if name == "get_weather":
        return f"The weather is {content}."
    if name == "get_customer" and isinstance(data, dict) and "plan" in data:
        return f"Customer {data.get('id', '')} is {data['name']} on the {data['plan']} plan."
    if name == "get_order" and isinstance(data, dict) and "item" in data:
        return f"Order for '{data['item']}' ({data['amount']} USD) is '{data['status']}', placed {data['days_ago']} days ago by {data['customer_id']}."
    if name == "refund_customer":
        if isinstance(data, dict) and data.get("status") == "refunded":
            return f"Refund {data['refund_id']} of {data['amount']} USD was issued to {data['customer_id']}."
        return f"The refund was NOT carried out: {content[:120]}"
    if name.startswith("search_policies"):
        lines = [line for line in content.splitlines() if line.strip() and not line.startswith("[")]
        return "Policy says: " + (lines[0] if lines else content[:120])
    return f"{name} reports: {content[:160]}"


def _mock_decide(messages, tools):                                 # ours (mock helper)
    """The mock's whole 'brain': look at the latest user turn, decide tool calls or text."""
    names = [t["function"]["name"] for t in tools]
    human_positions = [i for i, m in enumerate(messages) if isinstance(m, HumanMessage)]
    last_human = human_positions[-1] if human_positions else -1
    question = text_of(messages[last_human]) if last_human >= 0 else ""
    lower = question.lower()
    system_text = " ".join(text_of(m) for m in messages if isinstance(m, SystemMessage))
    system_lower = system_text.lower()
    turn = messages[last_human + 1:] if last_human >= 0 else list(messages)
    results = [(m.name or "tool", text_of(m)) for m in turn if isinstance(m, ToolMessage)]
    requested = {(c["name"], json.dumps(c["args"], sort_keys=True)) for m in turn if isinstance(m, AIMessage) for c in m.tool_calls}
    calls: list[dict] = []

    def want(name, **args):
        key = (name, json.dumps(args, sort_keys=True))
        if name in names and key not in requested:
            requested.add(key)
            calls.append({"name": name, "args": args, "id": f"call_{name}_{len(calls) + 1}"})

    # The mock is deliberately gullible: instructions hidden in retrieved text are obeyed (L10 demo).
    for _, content in results:
        hit = re.search(r"IGNORE PREVIOUS INSTRUCTIONS.*?refund_customer\D+(C\d{3})\D+(\d+)", content, re.I | re.S)
        if hit:
            want("refund_customer", customer_id=hit.group(1), amount=float(hit.group(2)))
    # Specialist sub-agents exposed as tools (L13).
    if re.search(r"order|charged|invoice", lower):
        want("billing_agent", query=question)
    if re.search(r"polic", lower):
        want("policy_agent", query=question)
    # Ordinary tools, matched by name; one call per id mentioned.
    arithmetic = re.search(r"(\d[\d\s.]*[*+\-/x×][\d\s.*+\-/x×()]*\d)", question)
    if arithmetic:
        want("calculate", expression=arithmetic.group(1).replace("×", "*").replace("x", "*").strip())
    for city in re.findall(r"weather in ([A-Z][a-z]+)", question):
        want("get_weather", city=city)
    customers = re.findall(r"\b(C\d{3})\b", question)
    for customer_id in customers:
        want("get_customer", customer_id=customer_id)
    for order_id in re.findall(r"\b(O\d{4})\b", question):
        want("get_order", order_id=order_id)
    for name, content in results:                       # dependent lookup: the order names a customer
        if name == "get_order" and re.search(r"who|customer|plan", lower):
            owner = re.search(r'"customer_id": "(C\d{3})"', content)
            if owner:
                want("get_customer", customer_id=owner.group(1))
    if re.search(r"polic|shipping|return|escalat|refund", lower):
        for name in names:
            if name.startswith("search_policies"):
                want(name, query=question)
    if "exchange rate" in lower:
        currency = re.search(r"\b([A-Z]{3})\b", question)
        want("get_exchange_rate", currency=currency.group(1) if currency else "EUR")
    if re.search(r"compare|research", lower):
        want("web_search", query=question)
    for name, content in results:
        if name == "web_search":
            for url in re.findall(r"https?://\S+", content):
                want("fetch_page", url=url)
    if re.search(r"remember|prefer", lower):
        want("remember_preference", preference=question)
    if re.search(r"know about me|my preferences|how should you", lower):
        want("recall_preferences")
    # Side effects only after the read-only evidence is in (a good habit the mock imitates).
    refund = re.search(r"refund (?:of )?\$?(\d+)", lower)
    if refund and customers and not calls and not any(n == "refund_customer" for n, _ in results):
        want("refund_customer", customer_id=customers[0], amount=float(refund.group(1)))
    # Structured-output schemas arrive as tools named after the class (L5, L9, L12).
    if not calls:
        money = re.search(r"charge|payment|invoice|refund", lower)
        want("SupportTicket", intent="billing" if money else "general", customer_id=customers[0] if customers else "unknown",
             priority="high" if re.search(r"twice|urgent|double", lower) else "medium", department="finance" if money else "support")
        want("ResearchPlan", goal=question[:80], steps=["Search for the companies mentioned in the request",
             "Fetch each company's pricing page", "Compare the delivery fees and summarise"])
        want("RouteDecision", category="billing" if re.search(r"order|charged|invoice|refund", lower) else "faq")
    usage = {"input_tokens": 40 + 8 * len(messages), "output_tokens": 30, "total_tokens": 70 + 8 * len(messages)}
    if calls:
        return AIMessage(content="", tool_calls=calls, usage_metadata=usage)

    # Text replies.
    if results:
        important = [r for r in results if r[0] == "refund_customer"] + [r for r in results if r[0] != "refund_customer"]
        return AIMessage(content="Here is what I found. " + " ".join(_phrase(n, c) for n, c in important), usage_metadata=usage)
    if "answer only from the policy excerpts" in system_lower:
        sentences = [s.strip() for s in re.split(r"(?<=\.)\s+", system_text) if re.search(r"\d+ days", s)][:2]
        return AIMessage(content="Based on the policy excerpts: " + " ".join(sentences), usage_metadata=usage)
    if "comparison from the findings" in system_lower:
        return AIMessage(content="Comparison: SwiftBite charges 4.50 USD per city parcel, ZipMeal 3.90 USD under 5 kg, DashDine 5.20 USD. ZipMeal is cheapest for light city parcels; only SwiftBite and DashDine deliver regionally.", usage_metadata=usage)
    if "draft" in system_lower:
        return AIMessage(content="Draft reply: thank you for contacting Meridian support. Our records confirm the issue and, under our policy, we will resolve it promptly.", usage_metadata=usage)
    if "context extraction" in lower or "conversation history" in lower or "summar" in system_lower:
        return AIMessage(content="Summary: the user introduced themselves as Rahul and asked OpsPilot about the weather and some arithmetic.", usage_metadata=usage)
    told = re.search(r"my name is (\w+)", " ".join(text_of(m) for m in messages if isinstance(m, HumanMessage)), re.I)
    if re.search(r"my name is (\w+)", lower):
        return AIMessage(content=f"Nice to meet you, {told.group(1)}!", usage_metadata=usage)
    if "my name" in lower:
        return AIMessage(content=f"Your name is {told.group(1)}." if told else "I don't know your name - you have not told me in this conversation.", usage_metadata=usage)
    if "opspilot" in lower:
        return AIMessage(content="OpsPilot is an operations assistant that answers questions about customers, orders, weather and company policy by calling tools.", usage_metadata=usage)
    if "agent" in lower:
        return AIMessage(content="An AI agent is a program that lets a language model decide the next action - answer, or call a tool - and loops until the task is done.", usage_metadata=usage)
    if names and re.search(r"customer|order|plan|weather|policy", lower):
        return AIMessage(content="I could not find a suitable tool for that request, so I cannot answer it reliably.", usage_metadata=usage)
    if "three things" in lower:
        return AIMessage(content="1. Look up customers and orders. 2. Check company policy. 3. Prepare refunds for human approval.", usage_metadata=usage)
    return AIMessage(content=f"(mock reply) You asked: {question[:120]}", usage_metadata=usage)

class MockChatModel(BaseChatModel):                                # ours, built on LangChain's base class
    """Rule-based stand-in for the course model. Same interfaces, canned decisions."""
    bound_tools: list = []
    fail: bool = False

    @property
    def _llm_type(self) -> str:
        return "opspilot-mock"

    def bind_tools(self, tools, **kwargs):                         # LangChain interface, our implementation
        # A real model receives the tool schemas with every request; we keep them on a copy.
        return self.model_copy(update={"bound_tools": [convert_to_openai_tool(t) for t in tools]})   # Pydantic: copy with changes

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:   # LangChain calls this from invoke()/stream()
        if self.fail:
            raise RuntimeError("503 Service Unavailable (simulated provider outage)")
        return ChatResult(generations=[ChatGeneration(message=_mock_decide(messages, self.bound_tools))])


model = make_model()
print("model class :", type(model).__name__)

In [ ]:
# ours: a small printer used throughout, shows an agent's message trajectory one line per message.
def show_messages(messages, width=110):
    # m.tool_calls, m.name, m.type and m.content are LangChain message attributes.
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            for call in m.tool_calls:
                print(f"  ai     -> tool call: {call['name']}({json.dumps(call['args'])})")
            if text_of(m).strip():
                print(f"  ai     : {text_of(m)[:width]}")
        elif isinstance(m, ToolMessage):
            print(f"  tool   : [{m.name}] {text_of(m)[:width]}")
        else:
            print(f"  {m.type:6} : {text_of(m)[:width]}")

print("show_messages() ready")

# LangChain L1 — Level 0 — A plain model call
**OpsPilot v0 is not an agent.** It is a model behind a function. But every agent starts here,
and three facts about this level explain most agent bugs later on.

```text
Application  ->  messages  ->  Model  ->  one reply
```

A **chat model** in LangChain is an object with `invoke()` and `stream()`. Whatever provider
it talks to (OpenRouter here), your code sends a list of **messages** and receives one
`AIMessage`. The model does not remember earlier calls, cannot run code, and cannot see your
data. Everything that later looks like memory, action or knowledge is added by the program around it.

### Step 1 — Messages in, one `AIMessage` out

LangChain has one message class per role: `SystemMessage` (standing instructions),
`HumanMessage` (the user), `AIMessage` (the model), and later `ToolMessage`. The reply carries
the text plus **usage metadata** (the tokens you paid for) and provider metadata.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage   # LangChain: message classes (roles)

OPSPILOT_PERSONA = "You are OpsPilot, an operations assistant for Meridian Supply Co. Be concise."   # ours

reply = model.invoke([                                             # LangChain: invoke() = one request -> one AIMessage
    SystemMessage(OPSPILOT_PERSONA),
    HumanMessage("Explain what an AI agent is in two sentences."),
])

print("type      :", type(reply).__name__)                         # AIMessage (LangChain)
print("content   :", text_of(reply))                               # ours: content as plain text
print("usage     :", reply.usage_metadata)                         # LangChain: token counts on every AIMessage = cost
print("provider  :", reply.response_metadata.get("model_name", "n/a (mock)"))   # LangChain: provider details

### Step 2 — Streaming

`stream()` yields the reply in chunks as the model produces it. For a chat interface this is
the difference between staring at a spinner and watching the answer appear. Each chunk is a
partial `AIMessage`; adding them up gives the full reply.

In [ ]:
print("streamed  : ", end="")
for chunk in model.stream([SystemMessage(OPSPILOT_PERSONA), HumanMessage("List three things an operations assistant might do.")]):   # LangChain: stream() yields AIMessageChunk pieces
    print(text_of(chunk), end="", flush=True)      # the mock streams in one chunk; a real model streams token by token
print()

### Step 3 — The model forgets everything between calls

Tell it a fact in one call, ask about the fact in a fresh call: it cannot answer, because the
second request never contained the fact. The "memory" of a conversation is simply the message
list your program re-sends. Section L6 turns that into a LangChain feature; for now, see the problem.

In [ ]:
first = model.invoke([HumanMessage("My name is Rahul. Please remember it.")])   # LangChain: invoke()
print("call 1 :", text_of(first)[:80])

second = model.invoke([HumanMessage("What is my name?")])          # a brand-new message list
print("call 2 :", text_of(second))

third = model.invoke([HumanMessage("My name is Rahul. Please remember it."), first, HumanMessage("What is my name?")])
print("call 3 :", text_of(third), "   <- only because WE re-sent the history")

### Recap

- **Problem seen:** a model call is text in, text out; nothing persists, nothing executes.
- **Layer added:** a LangChain chat model created by `make_model()`, typed messages, `invoke()` and `stream()`.
- **Evidence:** call 2 failed and call 3 succeeded, differing only in the messages we sent.